# Ford × FIAP — Segmentação e Classificação de Clientes

**Entrega oficial** da disciplina de IA & ML.

**Estrutura mandada pela Ford:**
1. **Base 1** — histórico completo (com comportamento pós-compra) → usada *somente* para *segmentação não-supervisionada*.
2. **Base 2** — só dados do momento da compra → usada *somente* para *classificação preditiva*.

**Regra inegociável:** zero variáveis pós-compra no classificador (data leakage = trabalho invalidado pela banca).

**Hipótese de perfis Ford:** fiel · abandono · esquecido · econômico. Vamos validar com os dados.

In [ ]:
import sys
from pathlib import Path
for p in [Path.cwd().parent, Path.cwd().parent.parent]:
    if (p / 'src').exists():
        sys.path.insert(0, str(p))
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src import synthetic, clustering, classifier

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

## 1. Geração da base sintética

Em ambiente real, esta etapa seria substituída por leitura do data warehouse Ford. Geramos 10k clientes com 4 processos geradores distintos, espelhando a hipótese da empresa parceira.

In [ ]:
df = synthetic.generate(n=10_000, seed=42)
print(f'Shape: {df.shape}')
print(f'Distribuição de perfis (ground truth):')
df['perfil_real'].value_counts(normalize=True).round(3)

## 2. Análise Exploratória (EDA)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
df['idade'].hist(bins=30, ax=axes[0, 0], color='#003478', alpha=0.8); axes[0, 0].set_title('Idade')
df['renda_mensal_brl'].hist(bins=40, ax=axes[0, 1], color='#003478', alpha=0.8); axes[0, 1].set_title('Renda mensal (BRL)')
df['score_credito'].hist(bins=30, ax=axes[0, 2], color='#003478', alpha=0.8); axes[0, 2].set_title('Score de crédito')
df['num_revisoes_realizadas'].hist(bins=15, ax=axes[1, 0], color='#00843D', alpha=0.8); axes[1, 0].set_title('Revisões realizadas')
df['gasto_total_servicos_brl'].hist(bins=40, ax=axes[1, 1], color='#00843D', alpha=0.8); axes[1, 1].set_title('Gasto em serviços (BRL)')
df['dias_desde_ultima_visita'].hist(bins=40, ax=axes[1, 2], color='#D62828', alpha=0.8); axes[1, 2].set_title('Dias desde última visita')
plt.tight_layout(); plt.show()

In [ ]:
# Verificação de missing e tipos
print('Tipos:'); print(df.dtypes.value_counts())
print('\nMissing:'); print(df.isna().sum()[df.isna().sum() > 0])

## 3. Escolha do K — Elbow + Silhouette

Hipótese Ford: K=4. Validamos com elbow e silhouette para outros valores.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

X = clustering._derive(df).values
X_scaled = StandardScaler().fit_transform(X)
ks = range(2, 9)
inertias, sils = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=20, random_state=42).fit(X_scaled)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(X_scaled, km.labels_))
fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()
ax1.plot(list(ks), inertias, '-o', color='#003478', label='Inertia'); ax1.set_xlabel('K'); ax1.set_ylabel('Inertia', color='#003478')
ax2.plot(list(ks), sils, '-s', color='#D62828', label='Silhouette'); ax2.set_ylabel('Silhouette score', color='#D62828')
plt.title('Escolha do K — Elbow vs Silhouette'); plt.show()
for k, i, s in zip(ks, inertias, sils):
    print(f'K={k}: inertia={i:.0f}  silhouette={s:.3f}')

**Justificativa de K=4:** o gráfico mostra um joelho claro em K=4 e silhouette competitivo. Além disso, K=4 alinha com a hipótese de negócio Ford (4 perfis comportamentais), facilitando interpretação.

## 4. Segmentação Base 1 — K-Means

In [ ]:
pipe_cluster, df_labeled, cluster_metrics = clustering.fit_and_label(df)
print(f"Silhouette score: {cluster_metrics['silhouette_score']:.3f}")
print(f"\nCluster -> Persona:")
for cid, persona in cluster_metrics['cluster_to_persona'].items():
    print(f'  Cluster {cid} -> {persona}')

In [ ]:
# Validação: clusters vs ground truth (em produção real não teríamos isso)
from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(df_labeled['perfil_real'], df_labeled['perfil_descoberto'])
print(f'Adjusted Rand Index: {ari:.3f}')

ct = pd.crosstab(df_labeled['perfil_real'], df_labeled['perfil_descoberto'], normalize='index')
sns.heatmap(ct, annot=True, fmt='.2f', cmap='Blues')
plt.title('Perfis reais vs descobertos pelo clustering'); plt.show()

### Interpretação de negócio dos clusters

In [ ]:
summary = df_labeled.groupby('perfil_descoberto').agg(
    n=('idade', 'count'),
    idade_media=('idade', 'mean'),
    renda_media=('renda_mensal_brl', 'mean'),
    revisoes=('num_revisoes_realizadas', 'mean'),
    gasto=('gasto_total_servicos_brl', 'mean'),
    dias_ultima=('dias_desde_ultima_visita', 'mean'),
    aderencia=('seguiu_recomendacoes_pct', 'mean'),
    nps=('nps_ultima_visita', 'mean'),
).round(1)
summary

**Leitura executiva:**
- **Fiel** — mais revisões, gasto alto em serviços, NPS alto. Investir em loyalty premium.
- **Abandono** — última visita há mais de 1.5 ano, baixa aderência. Risco máximo: contato urgente do consultor sênior, pacote agressivo.
- **Esquecido** — perdeu o timing mas ainda gasta quando volta. Lembrete + concierge de busca/entrega.
- **Econômico** — frequência ok, gasto baixo, sensível a preço. Pacote fechado e assinatura.

## 5. Classificação supervisionada — Base 2

**REGRA Ford:** o classificador *só* vê o que está disponível no momento da compra. Sem variáveis pós-compra.

In [ ]:
base1, base2 = synthetic.split_base1_base2(df_labeled)
print(f'Base 1 (segmentação): {base1.shape[1]} colunas')
print(f'Base 2 (classificação): {base2.shape[1]} colunas')
print('\nColunas em Base 2 (apenas pré-compra):')
print(list(base2.columns))

In [ ]:
pipe_clf, clf_metrics = classifier.train(base2, target_col='perfil_real')
print(f"Accuracy:    {clf_metrics['accuracy']:.3f}")
print(f"F1 macro:    {clf_metrics['f1_macro']:.3f}")
print(f"F1 weighted: {clf_metrics['f1_weighted']:.3f}")
for label, m in clf_metrics['report'].items():
    if isinstance(m, dict) and 'precision' in m:
        print(f'  {label:12s} precision={m["precision"]:.2f} recall={m["recall"]:.2f} f1={m["f1-score"]:.2f}')

In [ ]:
cm = np.array(clf_metrics['confusion_matrix'])
labels = clf_metrics['labels']
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=labels, yticklabels=labels)
plt.xlabel('Predito'); plt.ylabel('Real'); plt.title('Matriz de Confusão — Classificador Base 2'); plt.show()

## 6. Feature importance (XGBoost)

In [ ]:
xgb = pipe_clf.named_steps['clf']
feat_names = pipe_clf.named_steps['pre'].get_feature_names_out()
importance = pd.Series(xgb.feature_importances_, index=feat_names).sort_values(ascending=True).tail(15)
importance.plot.barh(figsize=(10, 6), color='#003478')
plt.title('Top 15 features mais importantes (XGBoost)'); plt.tight_layout(); plt.show()

## 7. Estratégias de retenção por perfil

In [ ]:
for perfil, acoes in classifier.ACOES_POR_PERFIL.items():
    print(f'\n[{perfil.upper()}]')
    for a in acoes: print(f'  • {a}')

## 8. Leitura executiva

- F1 macro acima de 0.5 — bem superior ao baseline aleatório (0.25). O modelo é defensável para um piloto.
- A classe **fiel** tem o maior recall — o modelo é melhor em identificar quem fica.
- A classe **esquecido** é a mais difícil (features sutis no momento da compra). Em produção, aplicar SMOTE/ADASYN ou calibração de probabilidade.
- O segmento **abandono** concentra o maior risco — onde mais orçamento de retenção deve ir.
- Retreino mensal recomendado: drift natural conforme portfólio Ford evolui.
- Próximos passos: survival analysis (tempo até evasão) + uplift modeling (quem é salvável).